In [20]:
!pip install --upgrade --quiet langchain-google-genai langchain-openai

# Remember, setup your free API Key using Google's AI Studio

https://aistudio.google.com/


In [27]:
!pip install -q langchain-groq


## Zero-Shot Prompting

Zero-shot prompting is the most basic form of interaction with an LLM. Its performance relies entirely on its pre-trained knowledge.

In "zero-shot," the word "shot" refers to the number of examples given to the AI model, so "zero-shot" means zero examples are provided.


In [30]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

# Load Groq API key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ")

# Use supported Groq model
llm = ChatGroq(model="llama-3.1-8b-instant")

prompt = "I started with 5 apples, bought 3 more, and ate 2. How many are left?"

completion = llm.invoke(prompt)
print(completion.content)


To find out how many apples are left, let's start with the initial number and add the ones you bought, then subtract the ones you ate.

Initial number: 5 apples
Apples bought: 3 apples
Total apples = Initial number + Apples bought = 5 + 3 = 8 apples

Now, subtract the apples you ate: 
Apples left = Total apples - Apples eaten = 8 - 2 = 6 apples

So, you have 6 apples left.


The model provides a direct answer, "6 apples," which is correct. However, the output lacks any reasoning or explanation. For more complex problems, this method is unreliable and can easily produce incorrect results without any way to debug the model's logic.

## Few-Shot Prompting

This technique provides the model with a few examples ("shots") to demonstrate the desired task and output format. This helps guide the model toward a more accurate response by showing it the pattern to follow.

In [31]:
# We provide examples of other math problems to guide the model.
prompt = """
Question: I had 10 pencils and gave 4 away. How many are left?
Answer: 6

Question: I bought 2 books on Monday and 5 on Tuesday. How many books did I buy?
Answer: 7

Question: I started with 5 apples, bought 3 more, and then ate 2. How many do I have left?
"""

In [32]:
completion = llm.invoke(prompt)
print(completion.content)

To find out how many apples are left, we need to add the initial number of apples (5) to the number of apples bought (3), and then subtract the number of apples eaten (2).

So, the calculation would be: 
5 (initial apples) + 3 (apples bought) = 8
8 - 2 (apples eaten) = 6

You have 6 apples left.


The output is again the correct number, "6". By providing examples, we increase the likelihood of getting the correct numerical answer and can better control the output format (e.g., "Answer: [number]"). However, it still doesn't reveal the underlying reasoning process, which is a significant drawback for complex tasks.

# Role-Based Prompting

Here, we assign a specific role or persona to the model to influence its tone, style, and even its problem-solving approach.

In [33]:
# We instruct the model to act as a specific persona.
prompt = """
Act as a math tutor explaining the solution to a student.
Question: I started with 5 apples, bought 3 more, and then ate 2. How many do I have left?
"""

In [34]:
completion = llm.invoke(prompt)
print(completion.content)

Let's break this down step by step.

You started with 5 apples. This is our initial number.

Then, you bought 3 more apples. To find out how many you have now, we need to add the number of apples you bought to the initial number.

So, we have 5 (initial apples) + 3 (apples bought) = 8 apples.

Now, you ate 2 apples. To find out how many apples you have left, we need to subtract the number of apples you ate from the total number of apples you had.

So, we have 8 (total apples) - 2 (apples eaten) = 6 apples.

Therefore, you have 6 apples left.


# Chain-of-Thought (CoT) Prompting

This technique explicitly instructs the model to break down the problem into intermediate steps before giving the final answer. This is often triggered by adding a simple phrase like "Let's think step-by-step".

In [35]:
# We append a phrase that triggers step-by-step reasoning.
prompt = """
Question: I started with 5 apples, bought 3 more, and then ate 2. How many do I have left?
Let's think step-by-step.
"""

In [36]:
completion = llm.invoke(prompt)
print(completion.content)

To find out how many apples you have left, let's break down the steps:

1. You started with 5 apples.
2. Then, you bought 3 more apples. Now, you have 5 (initial apples) + 3 (additional apples) = 8 apples.
3. After that, you ate 2 apples. Now, you have 8 (total apples after buying more) - 2 (apples eaten) = 6 apples.

So, you have 6 apples left.


The output is now highly structured and transparent. The model externalizes its reasoning process, showing each calculation explicitly. This is a major improvement for complex reasoning tasks, as it allows a user to verify the logic and easily spot any errors in the process.

# Prompt Chaining

Prompt chaining is a technique where a complex task is broken down into a series of smaller, sequential prompts. The output from one prompt is used as the input for the next, creating a "chain" that guides the model through a multi-step process.

Instead of asking the model to solve the entire problem in a single, complex request, you guide it through each logical step one by one. This approach enhances reliability, control, and transparency, as you can verify the output at each stage of the process.

In [37]:
# --- Prompt 1: First subtask (Addition) ---
prompt_1 = "I started with 5 apples and bought 3 more. How many apples do I have now?"
completion_1 = llm.invoke(prompt_1)
print(completion_1.content)

To find out how many apples you have now, you need to add the initial number of apples (5) to the number of apples you bought (3). 

5 (initial apples) + 3 (bought apples) = 8 apples

You now have 8 apples.


In [38]:
# --- Prompt 2: Second subtask (Subtraction) ---
prompt_2 = f"Following the previous calculation, I now have {completion_1.content.split()[-2]} apples and ate 2. How many do I have left?"
completion_2 = llm.invoke(prompt_2)
print(completion_2.content)

I didn't calculate anything previously. However, based on your new information, you started with 8 apples and ate 2. To find out how many you have left, we need to subtract 2 from 8.

8 (initial apples) - 2 (apples eaten) = 6

You have 6 apples left.


Prompt chaining provides the most explicit and controlled workflow.
- **Improvement:** This method is highly reliable because each step is simple and isolated. If an error occurs, it's easy to pinpoint exactly which subtask failed, making debugging much simpler than with a single, complex prompt. It gives the developer maximum control over the reasoning process.
- **Degradation:** The primary drawback is the increased complexity and latency. This approach requires multiple calls to the model, which is slower and can be more expensive than a single prompt. It also requires more effort from the developer to break down the task and manage the flow of information between prompts. For a simple problem like this, prompt chaining is overkill, but for complex, multi-stage tasks like data analysis or campaign planning, its benefits are significant.

# ReAct (Reasoning + Acting) Prompting

ReAct (**Re**asoning and **Act**ing) represents a paradigm shift in how LLMs can solve problems. It combines the internal reasoning of Chain of Thought with the ability to take Actions and process Observations from external tools or APIs. This creates a powerful feedback loop: the model **Thinks** about what it needs to know, performs an **Action** (like a search query or API call) to get that information, receives an **Observation** (the result of the action), and then uses that new information in its next step.

The primary benefit of this approach is its ability to ground the model's reasoning in external, factual, and up-to-date information.

This `Thought -> Action -> Observation` cycle is more than just a prompting technique; it is the fundamental foundation of an agentic AI.

Let's take a more complex example to look at ReAct Prompting.

In [39]:
# Mock function to simulate an external API call
def lookup_order_status(order_id: str) -> str:
    """Simulates looking up an order status from a database or API."""
    print(f"")
    statuses = {
        "ORD-12345": "Your order has been shipped and is expected to arrive on July 5th, 2025.",
        "ORD-67890": "Your order is currently being processed and has not yet shipped.",
        "ORD-11223": "We could not find an order with that ID. Please double-check the number."
    }
    return statuses.get(order_id, "Invalid order ID.")

In [40]:
# A new customer email requiring external tool use
customer_email = "Hi, I'm writing to check on the status of my recent order, #ORD-12345. Can you let me know where it is?"

# The ReAct prompt needs to be manually processed in a loop for this demonstration.
# Step 1: Simulated Initial Thought
react_prompt_1 = f"""
    You have access to a tool: `lookup_order_status(order_id: str)`.
    Given the user's query, decide if you need to use the tool.
    If so, respond with the tool call. If not, respond to the user directly.

    User Query: "{customer_email}"
    Thought: The user is asking for the status of order #ORD-12345. I need to use the `lookup_order_status` tool to get this information.
    Action: lookup_order_status(order_id='ORD-12345')
"""


# Step 2: Execute the Action and get the Observation
# In a real agent, this would be automated. Here, we call the function manually.
observation = lookup_order_status(order_id='ORD-12345')
print(f"\nObservation: {observation}")


# Step 3: Use the Observation to generate the final answer
react_prompt_2 = f"""
    You have performed an action and received an observation.
    Now, formulate the final response to the user.

    User Query: "{customer_email}"
    Action Taken: lookup_order_status(order_id='ORD-12345')
    Observation: "{observation}"

    Thought: The tool returned that the order has been shipped and provided a delivery date. I should now convey this information clearly and helpfully to the customer.
    Final Answer:
"""

final_response = llm.invoke(react_prompt_2)
print(final_response.content)



Observation: Your order has been shipped and is expected to arrive on July 5th, 2025.
"Hello, thank you for reaching out about the status of your recent order, #ORD-12345. We've checked on its status and we're happy to report that your order has been shipped and is now on its way to you. According to our tracking information, it's expected to arrive on July 5th, 2025. You should receive a separate shipping confirmation email with more details and tracking information shortly. If you have any other questions or concerns, please don't hesitate to ask."
